# ☀️ Solar Power Plant Analytics with Python
**Use case:** How are two solar plants really performing?  
**Data:** 15-minute readings from 2 plants (22 inverters each) + weather sensors, 15 May – 17 Jun 2020  
**Run the cells in order, top to bottom.** Each step says what you will see.

## Step 0 — Upload the four dataset files
Click **Choose Files** and select all four CSV files together.

In [ ]:
from google.colab import files
uploaded = files.upload()

needed = ["Plant_1_Generation_Data.csv", "Plant_1_Weather_Sensor_Data.csv",
          "Plant_2_Generation_Data.csv", "Plant_2_Weather_Sensor_Data.csv"]
missing = [f for f in needed if f not in uploaded]
print("✔ Upload complete" if not missing else f"✘ Missing files: {missing}")

## Step 1 — Load the tools

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
COLORS = {"Plant 1": "#2E96D0", "Plant 2": "#F2A33A"}   # sky blue and sun amber
CORAL = "#EF7B6B"
print("✔ Tools ready")

## Step 2 — Look at the raw data

In [ ]:
g1 = pd.read_csv("Plant_1_Generation_Data.csv");  w1 = pd.read_csv("Plant_1_Weather_Sensor_Data.csv")
g2 = pd.read_csv("Plant_2_Generation_Data.csv");  w2 = pd.read_csv("Plant_2_Weather_Sensor_Data.csv")

print(f"Plant 1: {len(g1):,} generation rows | {len(w1):,} weather rows | {g1.SOURCE_KEY.nunique()} inverters")
print(f"Plant 2: {len(g2):,} generation rows | {len(w2):,} weather rows | {g2.SOURCE_KEY.nunique()} inverters")
print("\nPlant 1 date format:", g1.DATE_TIME.iloc[0], "   Plant 2 date format:", g2.DATE_TIME.iloc[0])
print("\nGeneration data (one row = one inverter, every 15 minutes):")
display(g1.head(5))
print("Weather data (one row = one reading from the plant's sensor):")
display(w1.head(5))

## Step 3 — Clean and combine the data
Fix the date formats, give inverters short names, and stack both plants into one table.

In [ ]:
g1["DATE_TIME"] = pd.to_datetime(g1["DATE_TIME"], format="%d-%m-%Y %H:%M")   # Plant 1 is day-first
g2["DATE_TIME"] = pd.to_datetime(g2["DATE_TIME"])
w1["DATE_TIME"] = pd.to_datetime(w1["DATE_TIME"])
w2["DATE_TIME"] = pd.to_datetime(w2["DATE_TIME"])

g1["Plant"], g2["Plant"], w1["Plant"], w2["Plant"] = "Plant 1", "Plant 2", "Plant 1", "Plant 2"
gen = pd.concat([g1, g2], ignore_index=True).rename(columns={"SOURCE_KEY": "inverter"})
wx  = pd.concat([w1, w2], ignore_index=True)

# Long random inverter codes -> INV-01 ... INV-22 (numbered inside each plant)
gen["inverter"] = gen.groupby("Plant")["inverter"].transform(lambda s: s.astype("category").cat.codes + 1)
gen["inverter"] = "INV-" + gen["inverter"].astype(str).str.zfill(2)
gen["date"] = gen["DATE_TIME"].dt.normalize()

print("Date range:", gen.DATE_TIME.min(), "to", gen.DATE_TIME.max())
print("Missing values:", int(gen.isna().sum().sum()) + int(wx.isna().sum().sum()))
print("Rows per plant:", gen.Plant.value_counts().to_dict())

## Step 4 — Spot a data problem (units!)
An inverter turns DC power from the panels into AC power for the grid. A healthy inverter keeps about 97–99% of the power, so **AC ÷ DC should be close to 1**.

In [ ]:
run = gen[gen.DC_POWER > 0]
print("Median AC ÷ DC while running — BEFORE the fix:")
print((run.AC_POWER / run.DC_POWER).groupby(run.Plant).median().round(3))

# Plant 1 reports DC power in units that are 10x too large -> divide by 10
is_p1 = gen.Plant == "Plant 1"
gen.loc[is_p1, "DC_POWER"] = gen.loc[is_p1, "DC_POWER"] / 10

run = gen[gen.DC_POWER > 0]
print("\nMedian AC ÷ DC while running — AFTER the fix:")
print((run.AC_POWER / run.DC_POWER).groupby(run.Plant).median().round(3))

## Step 5 — Create the new columns
The data has no energy or efficiency columns. We build them, then a plant-level table that joins power with weather.

In [ ]:
gen["energy_kwh"] = gen["AC_POWER"] * 0.25                     # kW for 15 min = kW x 0.25 hours
gen["efficiency"] = np.where(gen.DC_POWER > 0, gen.AC_POWER / gen.DC_POWER * 100, np.nan)

# Plant-level table: average inverter x number of inverters, so a missing reading
# does not make the plant look weaker than it was.
n_inv = gen.groupby("Plant").inverter.nunique()
plant15 = (gen.groupby(["Plant", "DATE_TIME"])
              .agg(avg_ac=("AC_POWER", "mean"), avg_dc=("DC_POWER", "mean")).reset_index())
plant15["ac_kw"] = plant15.avg_ac * plant15.Plant.map(n_inv)
plant15["dc_kw"] = plant15.avg_dc * plant15.Plant.map(n_inv)
plant15 = plant15.drop(columns=["avg_ac", "avg_dc"]).merge(
    wx[["Plant", "DATE_TIME", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]],
    on=["Plant", "DATE_TIME"], how="left")
plant15 = plant15.rename(columns={"AMBIENT_TEMPERATURE": "temp_amb", "MODULE_TEMPERATURE": "temp_mod",
                                  "IRRADIATION": "irradiation"})
plant15["date"] = plant15.DATE_TIME.dt.normalize()
plant15["hour"] = plant15.DATE_TIME.dt.hour

print("New columns on inverter data:", ["energy_kwh", "efficiency"])
display(gen[["Plant", "inverter", "DATE_TIME", "AC_POWER", "energy_kwh", "efficiency"]].iloc[40:46])
print("Plant-level table:")
display(plant15.head(3))

## Step 6 — Q1: How much energy does each plant produce?

In [ ]:
daily = (plant15.groupby(["Plant", "date"]).ac_kw.sum() * 0.25 / 1000).rename("energy_mwh").reset_index()
daily["avg7"] = daily.groupby("Plant").energy_mwh.transform(lambda s: s.rolling(7, min_periods=1).mean())

fig, ax = plt.subplots(figsize=(11, 4))
for plant, color in COLORS.items():
    d = daily[daily.Plant == plant]
    ax.plot(d.date, d.energy_mwh, color=color, marker="o", ms=3, alpha=.45, lw=1)
    ax.plot(d.date, d.avg7, color=color, lw=3, label=f"{plant} (7-day average)")
ax.set_title("Daily energy produced (MWh)", fontweight="bold"); ax.set_ylabel("MWh per day")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.legend(); plt.tight_layout(); plt.show()

best  = daily.loc[daily.groupby("Plant").energy_mwh.idxmax()].set_index("Plant")
worst = daily.loc[daily.groupby("Plant").energy_mwh.idxmin()].set_index("Plant")
summary = daily.groupby("Plant").energy_mwh.agg(total_mwh="sum", avg_mwh_per_day="mean").round(1)
summary["best_day"]  = best.date.dt.strftime("%d %b")
summary["best_mwh"]  = best.energy_mwh.round(1)
summary["worst_day"] = worst.date.dt.strftime("%d %b")
summary["worst_mwh"] = worst.energy_mwh.round(1)
display(summary)

## Step 7 — Q2: When during the day do the plants produce power?

In [ ]:
hourly = plant15.groupby(["Plant", "hour"]).ac_kw.mean().unstack(0) / 1000     # MW

fig, ax = plt.subplots(figsize=(9, 3.8))
for plant, color in COLORS.items():
    ax.fill_between(hourly.index, hourly[plant], color=color, alpha=.25)
    ax.plot(hourly.index, hourly[plant], color=color, lw=2.5, label=plant)
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("Hour of day"); ax.set_ylabel("Average AC power (MW)")
ax.set_title("Average daily power curve", fontweight="bold"); ax.legend()
plt.tight_layout(); plt.show()

for plant in COLORS:
    on = hourly[plant][hourly[plant] > 0.05 * hourly[plant].max()]
    print(f"{plant}: peak at {hourly[plant].idxmax()}:00 ({hourly[plant].max():.1f} MW) | "
          f"produces from about {on.index.min()}:00 to {on.index.max()}:00")

## Step 8 — Q3: How does the weather drive power?
Left: sunshine (irradiation) vs power. Right: does a hot panel produce less per unit of sunshine?

In [ ]:
day = plant15.dropna(subset=["irradiation"]).copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for plant, color in COLORS.items():
    d = day[day.Plant == plant]
    axes[0].scatter(d.irradiation, d.ac_kw / 1000, s=6, alpha=.35, color=color,
                    label=f"{plant} (r = {d.irradiation.corr(d.ac_kw):.2f})")
axes[0].set_xlabel("Irradiation (kW/m²)"); axes[0].set_ylabel("Plant AC power (MW)")
axes[0].set_title("More sunshine = more power", fontweight="bold"); axes[0].legend()

sun = day[day.irradiation >= 0.3].copy()
sun["output_per_sun"] = sun.ac_kw / sun.irradiation / 1000          # MW per (kW/m²)
sun["panel_temp"] = pd.cut(sun.temp_mod, [0, 35, 45, 55, 100], labels=["< 35°C", "35–45°C", "45–55°C", "> 55°C"])
band = sun.groupby(["Plant", "panel_temp"], observed=True).output_per_sun.mean().reset_index()
sns.barplot(data=band, x="panel_temp", y="output_per_sun", hue="Plant", palette=COLORS, ax=axes[1])
axes[1].set_xlabel("Panel (module) temperature"); axes[1].set_ylabel("MW per unit of sunshine")
axes[1].set_title("Heat effect on output", fontweight="bold"); axes[1].set_ylim(0, 38)
plt.tight_layout(); plt.show()

print("Correlation of plant AC power with each weather reading:")
display(day.groupby("Plant")[["irradiation", "temp_mod", "temp_amb"]]
           .apply(lambda d: d.corrwith(day.loc[d.index, "ac_kw"])).round(2))
print("Output per unit of sunshine (MW) by panel temperature:")
display(band.pivot(index="panel_temp", columns="Plant", values="output_per_sun").round(1))

## Step 9 — Q4: Which inverters are underperforming?
Every plant has 22 inverters that see the same sun, so they should produce about the same energy. We compare each one to the plant's median inverter.

In [ ]:
inv_day = gen.groupby(["Plant", "inverter", "date"]).energy_kwh.sum().reset_index()
inv_total = (inv_day.groupby(["Plant", "inverter"]).energy_kwh.sum() / 1000).rename("energy_mwh").reset_index()
inv_total["vs_median"] = inv_total.energy_mwh / inv_total.groupby("Plant").energy_mwh.transform("median") * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharex=True)
for ax, (plant, color) in zip(axes, COLORS.items()):
    d = inv_total[inv_total.Plant == plant].sort_values("energy_mwh")
    ax.barh(d.inverter, d.energy_mwh, color=[CORAL if v < 85 else color for v in d.vs_median])
    ax.set_title(f"{plant}: energy per inverter (MWh)", fontweight="bold")
plt.tight_layout(); plt.show()

under = inv_total[inv_total.vs_median < 85].sort_values("vs_median")
print("Inverters below 85% of their plant's median (coral bars):")
display(under.round(1))

In [ ]:
inv_day["rel"] = inv_day.energy_kwh / inv_day.groupby(["Plant", "date"]).energy_kwh.transform("median") * 100
cmap = sns.blend_palette([CORAL, "#FFFFFF", "#2E96D0"], as_cmap=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 8.5))
for ax, plant in zip(axes, COLORS):
    pv = inv_day[inv_day.Plant == plant].pivot(index="inverter", columns="date", values="rel")
    pv.columns = [c.strftime("%d %b") for c in pv.columns]
    sns.heatmap(pv, cmap=cmap, center=100, vmin=0, vmax=130, ax=ax, yticklabels=True, cbar_kws={"label": "% of median inverter"})
    ax.set_title(f"{plant}: daily energy vs the median inverter (coral = weak)", fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(axis="y", labelsize=8, rotation=0)
plt.tight_layout(); plt.show()

## Step 10 — Q5: Did anything go wrong on particular days?
A cloudy day is not a fault. So we divide each day's energy by that day's sunshine. If **output per sun** drops, something other than weather happened.

In [ ]:
sun_day = (plant15.groupby(["Plant", "date"]).irradiation.sum() * 0.25).rename("sun_kwh_m2").reset_index()
daily = daily.drop(columns=[c for c in ["sun_kwh_m2", "mwh_per_sun", "vs_median"] if c in daily]).merge(sun_day, on=["Plant", "date"])
daily["mwh_per_sun"] = daily.energy_mwh / daily.sun_kwh_m2
daily["vs_median"] = daily.mwh_per_sun / daily.groupby("Plant").mwh_per_sun.transform("median") * 100

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for plant, color in COLORS.items():
    d = daily[daily.Plant == plant]
    axes[0].plot(d.date, d.sun_kwh_m2, color=color, marker="o", ms=3, label=plant)
    bad = d[d.vs_median < 85]
    axes[1].plot(d.date, d.vs_median, color=color, marker="o", ms=3, label=plant)
    axes[1].scatter(bad.date, bad.vs_median, color=CORAL, s=60, zorder=5)
axes[0].set_title("Sunshine received per day (kWh/m²)", fontweight="bold"); axes[0].legend()
axes[1].axhline(85, color=CORAL, ls="--"); axes[1].text(daily.date.min(), 86, "85% alert line", color=CORAL)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[1].set_title("Output per sun vs normal (100 = normal) — coral dots = suspicious days", fontweight="bold")
plt.tight_layout(); plt.show()

flag = daily[daily.vs_median < 85].sort_values("vs_median")
print("Days where the plant produced much less than the sunshine explains:")
display(flag[["Plant", "date", "sun_kwh_m2", "energy_mwh", "vs_median"]].round(1))

## Step 11 — Q6: Live inverter checker
We learn what a *normal* inverter produces at each sunshine level. Then we compare any new reading with it.

In [ ]:
inv_w = gen.merge(wx[["Plant", "DATE_TIME", "IRRADIATION", "MODULE_TEMPERATURE"]],
                  on=["Plant", "DATE_TIME"], how="inner")
inv_w["irr_bin"] = (inv_w.IRRADIATION / 0.1).round() * 0.1
curve = inv_w.groupby(["Plant", "irr_bin"]).AC_POWER.agg(["median", "size"]).reset_index()
curve = curve[curve["size"] >= 50]
CURVES = {p: (d.irr_bin.values, d["median"].values) for p, d in curve.groupby("Plant")}

def check_inverter(plant, irradiation, ac_kw):
    expected = float(np.interp(irradiation, *CURVES[plant]))
    if expected < 20:
        return dict(status="LOW SUN", icon="🌙", expected=expected, ratio=None,
                    action="Too little sun to judge this inverter.")
    ratio = ac_kw / expected * 100
    if ratio >= 85: return dict(status="OK", icon="✅", expected=expected, ratio=ratio, action="Working normally.")
    if ratio >= 60: return dict(status="WARNING", icon="⚠️", expected=expected, ratio=ratio, action="Schedule an inspection.")
    return dict(status="ALERT", icon="🚨", expected=expected, ratio=ratio, action="ALERT — send a technician now.")

for args in [("Plant 1", 0.8, 1000), ("Plant 2", 0.8, 300)]:
    r = check_inverter(*args)
    print(args, "->", r["icon"], r["status"], f"(expected {r['expected']:.0f} kW)")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

plant_dd = widgets.Dropdown(options=["Plant 1", "Plant 2"], description="Plant:")
irr_box  = widgets.FloatText(value=0.8, description="Sun (kW/m²):")
ac_box   = widgets.FloatText(value=900, description="AC power (kW):")
button   = widgets.Button(description="Check inverter", button_style="success")
out      = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        r = check_inverter(plant_dd.value, irr_box.value, ac_box.value)
        color = {"OK": "#E3F6EF", "WARNING": "#FFF3DC", "ALERT": "#FDE7E4", "LOW SUN": "#EEF4F6"}[r["status"]]
        pct = "" if r["ratio"] is None else f" — producing {r['ratio']:.0f}% of normal"
        display(HTML(f"<div style='padding:12px;border-radius:8px;background:{color};color:#1B2A4A'>"
                     f"<b>{r['icon']} {r['status']}</b>{pct}<br>Expected at this sunshine: {r['expected']:.0f} kW<br>{r['action']}</div>"))
button.on_click(on_click)
display(widgets.VBox([plant_dd, irr_box, ac_box, button, out]))

## Step 12 — Live inverter feed
Eight readings arrive one by one. Each is checked the moment it arrives, and a 🚨 ALERT appears for faulty ones.

In [ ]:
pool = inv_w[inv_w.IRRADIATION >= 0.5].copy()
pool["expected"] = [np.interp(i, *CURVES[p]) for p, i in zip(pool.Plant, pool.IRRADIATION)]
pool["ratio"] = pool.AC_POWER / pool.expected * 100

# A realistic mix: 5 healthy readings and 3 faulty ones, shuffled
feed = pd.concat([pool[pool.ratio >= 85].sample(5, random_state=4),
                  pool[pool.ratio < 60].sample(3, random_state=4)]).sample(frac=1, random_state=1)

counts = {"OK": 0, "WARNING": 0, "ALERT": 0}
for _, r in feed.iterrows():
    res = check_inverter(r.Plant, r.IRRADIATION, r.AC_POWER)
    counts[res["status"]] += 1
    print(f"{r.DATE_TIME:%d %b %H:%M} | {r.Plant} {r.inverter} | sun {r.IRRADIATION:.2f} | "
          f"{r.AC_POWER:6.0f} kW (expected {res['expected']:.0f}) | {res['icon']} {res['status']}")
    time.sleep(1.5)
print("\nSummary:", counts)

## Step 13 — Bonus: forecast power with machine learning (optional)
Can weather alone predict how much power a plant will make? We train on the first weeks and test on the **last 7 days**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

features = ["irradiation", "temp_mod", "temp_amb", "hour"]
ml = plant15.dropna(subset=features)
cut = ml.date.max() - pd.Timedelta(days=6)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
rows = []
for ax, (plant, color) in zip(axes, COLORS.items()):
    d = ml[ml.Plant == plant]
    train, test = d[d.date < cut], d[d.date >= cut]
    model = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1).fit(train[features], train.ac_kw)
    pred = model.predict(test[features])
    rows.append([plant, round(mean_absolute_error(test.ac_kw, pred) / 1000, 2), round(r2_score(test.ac_kw, pred), 3)])
    ax.plot(test.DATE_TIME, test.ac_kw / 1000, color=color, lw=1.8, label="Actual")
    ax.plot(test.DATE_TIME, pred / 1000, "k--", lw=1, label="Predicted")
    ax.set_title(f"{plant}: last 7 days (MW)", fontweight="bold"); ax.legend()
    ax.xaxis.set_major_locator(mdates.DayLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows, columns=["Plant", "Average error (MW)", "R² score"]))

## Step 14 — Save results for Excel / Power BI / Tableau

In [ ]:
daily.to_csv("daily_energy_summary.csv", index=False)
inv_total.to_csv("inverter_performance.csv", index=False)
plant15.to_csv("plant_15min_with_weather.csv", index=False)

for f in ["daily_energy_summary.csv", "inverter_performance.csv", "plant_15min_with_weather.csv"]:
    files.download(f)
print("✔ 3 files saved")